In [1]:
# Assignment 6 Pt.2 
# Shawn Tribuce
# DS 4200

The dataset I selected was from a previous class and pertains to a topic that piques
my immense curiosity is the World Happiness Report. Personally, I'm intrigued by whether
money contributes to happiness (are the wealthiest countries the happiest) and 
which factors influence the overall score. My source is the World Happiness Report,
an annual publication by the United Nations Sustainabale Development Solutions Network.
I imported the data using Kaggle and it encompasses 155 countries spanning the years 
2015, 2016, and 2017. Upon examining the data in Excel, I observed that Nordic countries
tend to be the happiest, while Sub-Saharan African nations cluster at the bottom. 
Additionally, I noticed that generosity and trust in the government are the weakest 
contributors due to their high variance and low averages. 


In [2]:
!pip install kagglehub

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("unsdsn/world-happiness")

print("Path to dataset files:", path)

Path to dataset files: /Users/shawntribuce/.cache/kagglehub/datasets/unsdsn/world-happiness/versions/2


In [4]:
import pandas as pd

df15 = pd.read_csv('/Users/shawntribuce/.cache/kagglehub/datasets/unsdsn/world-happiness/versions/2/2015.csv')
df16 = pd.read_csv('/Users/shawntribuce/.cache/kagglehub/datasets/unsdsn/world-happiness/versions/2/2016.csv')
df17 = pd.read_csv('/Users/shawntribuce/.cache/kagglehub/datasets/unsdsn/world-happiness/versions/2/2017.csv')

print("2015:", df15.columns.tolist())
print("2016:", df16.columns.tolist())
print("2017:", df17.columns.tolist())
print("\n2015 regions:", df15['Region'].unique())

2015: ['Country', 'Region', 'Happiness Rank', 'Happiness Score', 'Standard Error', 'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)', 'Generosity', 'Dystopia Residual']
2016: ['Country', 'Region', 'Happiness Rank', 'Happiness Score', 'Lower Confidence Interval', 'Upper Confidence Interval', 'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)', 'Freedom', 'Trust (Government Corruption)', 'Generosity', 'Dystopia Residual']
2017: ['Country', 'Happiness.Rank', 'Happiness.Score', 'Whisker.high', 'Whisker.low', 'Economy..GDP.per.Capita.', 'Family', 'Health..Life.Expectancy.', 'Freedom', 'Generosity', 'Trust..Government.Corruption.', 'Dystopia.Residual']

2015 regions: ['Western Europe' 'North America' 'Australia and New Zealand'
 'Middle East and Northern Africa' 'Latin America and Caribbean'
 'Southeastern Asia' 'Central and Eastern Europe' 'Eastern Asia'
 'Sub-Saharan Africa' 'Southern Asia']


In [5]:
import pandas as pd

# load the data
df15 = pd.read_csv('/Users/shawntribuce/.cache/kagglehub/datasets/unsdsn/world-happiness/versions/2/2015.csv')
df16 = pd.read_csv('/Users/shawntribuce/.cache/kagglehub/datasets/unsdsn/world-happiness/versions/2/2016.csv')
df17 = pd.read_csv('/Users/shawntribuce/.cache/kagglehub/datasets/unsdsn/world-happiness/versions/2/2017.csv')

# standardize 2017 column names to combine with continent
df17 = df17.rename(columns={
    'Happiness.Rank': 'Happiness Rank',
    'Happiness.Score': 'Happiness Score',
    'Economy..GDP.per.Capita.': 'Economy (GDP per Capita)',
    'Health..Life.Expectancy.': 'Health (Life Expectancy)',
    'Trust..Government.Corruption.': 'Trust (Government Corruption)'
})

# add year column
df15['Year'] = 2015
df16['Year'] = 2016
df17['Year'] = 2017

# merge Region into 2017 from 2015
region_map = df15[['Country', 'Region']].drop_duplicates()
df17 = df17.merge(region_map, on='Country', how='left')

# keep only shared columns
cols = ['Country', 'Region', 'Year', 'Happiness Rank', 'Happiness Score',
        'Economy (GDP per Capita)', 'Family', 'Health (Life Expectancy)',
        'Freedom', 'Trust (Government Corruption)', 'Generosity']

df15 = df15[cols]
df16 = df16[cols]
df17 = df17[cols]

# combine the dataframes
df = pd.concat([df15, df16, df17], ignore_index=True)

In [6]:
# manually assign missing regions since didn't auto assign
region_fixes = {
    'Taiwan Province of China': 'Eastern Asia',
    'Hong Kong S.A.R., China': 'Eastern Asia',
    'Belize': 'Latin America and Caribbean',
    'Somalia': 'Sub-Saharan Africa',
    'Namibia': 'Sub-Saharan Africa',
    'South Sudan': 'Sub-Saharan Africa'
}

df['Region'] = df.apply(
    lambda row: region_fixes.get(row['Country'], row['Region']), axis=1
)

# verify no more nulls
print(df[df['Region'].isna()]['Country'].unique())
print(df.shape)

[]
(470, 11)


In [7]:
!pip install pycountry

The map below provides an overview of happiness scores across 155 countries in 2017.
Darker colors represent lower happiness levels, while lighter yellow indicates higher
happiness levels. Gray indicates the absence of data for that year. I specifically 
chose the sequential colors yellow and red because yellow is naturally associated with
happiness, while red is associated with anger or negative feelings. I opted for 
diverging schemes because there are no significant happiness scores. For my 
interactivity, I included a radio button to filter one continent at a time, making all 
other continents gray. This technique, known as focus and context keeping, ensures that
the full map remains visible while still highlighting the interested region. Looking 
at the data, North America and Western Europe are visibly the lightest/happiest regions,
while Sub-Saharan Africa is the darkest region in the world. Australia and New Zealand
appear to be the happiest countries in the Asia-Pacific region. The data seems to be 
skewed positively towards European decent countries, which makes me question the 
integrity of this study. 

In [8]:
# Visualization 1: World Happiness Choropleth Map 

import pycountry
import altair as alt
from vega_datasets import data

# load world topojson for geographic shapes
world = alt.topo_feature(data.world_110m.url, 'countries')

# function to convert country name to numeric ISO code
def get_country_id(name):
    try:
        return int(pycountry.countries.lookup(name).numeric)
    except:
        return None

# filter to 2017 and assign numeric country IDs
df_map = df[df['Year'] == 2017].copy()
df_map['id'] = df_map['Country'].apply(get_country_id)

# radio button filter to isolate one continent at a time
region_radio = alt.binding_radio(
    options=[None, 'Europe', 'Americas', 'Asia-Pacific', 'Middle East & Africa'],
    labels=['All', 'Europe', 'Americas', 'Asia-Pacific', 'Middle East & Africa'],
    name='Filter by Continent: '
)

# selection tied to Continent field
region_select = alt.selection_point(
    fields=['Continent'],
    bind=region_radio
)

# base layer to fill gray for all countries including those with no data
map_base = alt.Chart(world).mark_geoshape(
    fill='#d3d3d3',
    stroke='white',
    strokeWidth=0.5
)

# color layer to join happiness data by country ID and apply color
map_color = alt.Chart(world).mark_geoshape(
    stroke='white',
    strokeWidth=0.5
).transform_lookup(
    lookup='id',
    from_=alt.LookupData(df_map, 'id', ['Happiness Score', 'Country', 'Region', 'Continent'])
).encode(
    # unselected continents grays out
    color=alt.condition(
        region_select,
        alt.Color(
            'Happiness Score:Q',
            scale=alt.Scale(scheme='darkred', domain=[2.5, 8]),
            legend=alt.Legend(title='Happiness Score')
        ),
        alt.value('lightgray')
    ),
    tooltip=['Country:N', 'Happiness Score:Q', 'Continent:N']
).add_params(
    region_select
).project('naturalEarth1')

# final map visualizaiton
map_chart = (map_base + map_color).properties(
    width=750,
    height=450,
    title='World Global Happiness Scores in 2017'
).project('naturalEarth1')

map_chart

alt.LayerChart(...)

In [9]:
# Assign the regions in the dataset to each continent for the following data set.
continent_map = {
    'Western Europe': 'Europe',
    'Central and Eastern Europe': 'Europe',
    'North America': 'Americas',
    'Latin America and Caribbean': 'Americas',
    'Australia and New Zealand': 'Asia-Pacific',
    'Southeastern Asia': 'Asia-Pacific',
    'Eastern Asia': 'Asia-Pacific',
    'Southern Asia': 'Asia-Pacific',
    'Middle East and Northern Africa': 'Middle East & Africa',
    'Sub-Saharan Africa': 'Middle East & Africa'
}

df['Continent'] = df['Region'].map(continent_map)

The bubble plot below simultaneously encodes three variables: GDP per Capita on the
x-axis, Happiness Score on the y-axis, and life expectancy as bubble size. Each bubble
represents a country, colored by its binned continent. For this visualization, I chose
a colorblind-friendly palette to ensure accessibility for everyone. Additionally, I 
attempted to distinguish each continent with unique colors to avoid color dominance 
or overshadowing. I also implemented a GDP threshold filter to highlight regions of 
the graph that the user would want to focus on. Furthermore, I allowed users to click
on any bubble, which would highlight the corresponding continent. I observed a clear 
positive correlation between GDP per Capita and Happiness Score, with European countries
clustered at the top right. Not only is GDP higher, but life expectancy is also longer 
in happier countries. Surprisingly, Latin American countries had a higher Happiness 
Score despite lower GDP, suggesting that money isn't the sole determinant of happiness. 

In [10]:
# Visualization 2: Bubble Plot
 
df_bubble = df[df['Year'] == 2017].copy()

continent_colors = alt.Scale(
    domain=['Europe', 'Americas', 'Asia-Pacific', 'Middle East & Africa'],
    range=['#0072B2', '#E69F00', '#009E73', '#D55E00']
)

brush = alt.selection_point(fields=['Continent'])

# slider parameter
gdp_slider = alt.binding_range(
    min=0, max=2.0, step=0.05, name='GDP Threshold: '
)
gdp_param = alt.param(bind=gdp_slider, value=2.0)

bubble_chart = alt.Chart(df_bubble).mark_circle(opacity=0.7).encode(
    x=alt.X('Economy (GDP per Capita):Q', title='GDP per Capita',
        scale=alt.Scale(zero=False)),
    y=alt.Y('Happiness Score:Q', title='Happiness Score',
        scale=alt.Scale(domain=[2.5, 8])),
    size=alt.Size('Health (Life Expectancy):Q', title='Life Expectancy',
        scale=alt.Scale(range=[50, 500])),
    color=alt.condition(
        brush,
        alt.Color('Continent:N', scale=continent_colors,
            legend=alt.Legend(title='Continent')),
        alt.value('lightgray')
    ),
    opacity=alt.condition(
        alt.datum['Economy (GDP per Capita)'] <= gdp_param,
        alt.value(0.8),
        alt.value(0.1)
    ),
    tooltip=[
        'Country:N', 'Continent:N', 'Happiness Score:Q',
        'Economy (GDP per Capita):Q', 'Health (Life Expectancy):Q'
    ]
).add_params(
    brush, gdp_param
).properties(
    width=700,
    height=450,
    title='Happiness Score vs. GDP per Capita by Continent in 2017'
).configure_axis(grid=False).configure_view(strokeWidth=0)

bubble_chart

alt.Chart(...)


The ridgeline plot below displays the distribution of happiness scores for each of
the continents overlapping my density curves. I chose this specific unusual graphic
because it showcases the distribution of the happiness score to see if it was highly
skewed or if there were outliers. I also chose to use a color-friendly palette in the
visualization so that it can be distinguishable to any viewer. If you hover on any of
the curves, it grayed out the other maps below and above it to focus on the continent
you would like. I believe this was a great indication of the distribution because you
can see that the Americas have some outliers on the lower scale but is highly 
right-skewed. While the Middle East, Africa, and Asia have a more bimodal shape with
a lot of data points clustered around the average. Europe, by far, has the most 
variation in its scores with a lot of range between 4 and 8, which I found particularly
interesting. 

In [11]:
# Visualization 3: Ridgeline Plot

df_ridge = df[df['Year'] == 2017].copy()

continent_colors = alt.Scale(
    domain=['Europe', 'Americas', 'Asia-Pacific', 'Middle East & Africa'],
    range=['#0072B2', '#E69F00', '#009E73', '#D55E00']
)

step = 40
overlap = 2

# hover to highlight a continent's distribution
highlight = alt.selection_point(
    fields=['Continent'],
    on='mouseover'
)

ridgeline = alt.Chart(df_ridge).transform_density(
    density='Happiness Score',
    as_=['Happiness Score', 'density'],
    groupby=['Continent'],
    extent=[2, 9]
).mark_area(
    interpolate='monotone',
    stroke='white',
    strokeWidth=0.5
).encode(
    x=alt.X(
        'Happiness Score:Q',
        title='Happiness Score'
    ),
    y=alt.Y(
        'density:Q',
        title=None,
        axis=None,
        scale=alt.Scale(range=[step, -step * overlap])
    ),
    color=alt.Color(
        'Continent:N',
        scale=continent_colors,
        legend=alt.Legend(title='Continent')
    ),
    # hovering a continent brightens it, dims others
    fillOpacity=alt.condition(highlight, alt.value(0.9), alt.value(0.3)),
    row=alt.Row(
        'Continent:N',
        title=None,
        header=alt.Header(
            labelAngle=0,
            labelAlign='right',
            labelFontSize=11
        )
    )
).add_params(
    highlight
).properties(
    width=600,
    height=step,
    title='Global Distribution of Happiness Scores by Continent in 2017'
).configure_facet(
    spacing=0
).configure_view(
    stroke=None
).configure_axis(
    grid=False
)

ridgeline

alt.Chart(...)

Conclusion 

The World Happiness Report data reveals that Europe and North America consistently rank as the happiest regions, 
while Sub-Saharan Africa scores the lowest. However, there is significant variation within each continent. The bubble 
chart shows that although GDP is the strongest indicator of happiness for citizens in that region, Latin America stands
out as an outlier, suggesting that more money doesn't always equate to happiness. One conclusion I draw is that wealth 
matters, but freedom, social support, and the absence of corruption in a country's government can explain why countries
with similar GDP levels experience different levels of happiness. I would also like to point out that the gap between 
the happiest and unhappiest countries is roughly 5 points on a 10-point scale, which is quite striking and worth noting.

In [12]:
import os
os.system('jupyter nbconvert --to html HW6_Pt2.ipynb')

[NbConvertApp] Converting notebook HW6_Pt2.ipynb to html
[NbConvertApp] Writing 326034 bytes to HW6_Pt2.html


0